In [14]:
!pip install groq

In [15]:
import groq
import os
import ast
import re
import json
from datetime import datetime

In [16]:
os.environ["GROQ_API_KEY"] = "YOUR_GROQ_API_KEY"
client = groq.Groq(api_key=os.environ["GROQ_API_KEY"])

In [17]:
def get_completion(prompt, model="openai/gpt-oss-120b", temperature=0.7):
    response = client.chat.completions.create(
        model=model,
        messages=[{"role":"user", "content": prompt}],
        temperature=temperature,
    )
    return response.choices[0].message.content

In [18]:
def planner_agent(topic):
  prompt = f"""
  You are a planning agent responsible for organizing a research flow.

  Your job is to write a clear, step-by-step research plan as a valid Python list,
  where each step is a string.

  Each step should be atomic and executable by one of these agents:
  - A research agent who can search for information.
  - A writer agent who can draft research summaries.
  - An editor agent who can reflect and revise drafts.

  Rules:
  - DO NOT include irrelevant tasks like "create CSV", "set up a repo",
  "install packages".
  - DO include real research tasks (search, summarize, draft, revise).
  - DO NOT include explanation text - return only the Python list.
  - The final step should be to write the complete research report.
  - Limit to 4 steps maximum

  Topic: "{topic}"
  """
  response = get_completion(prompt, temperature=1.0)

  # Robustly remove markdown code blocks from the response
  response = response.strip()
  # Remove leading '```python' or '```'
  response = re.sub(r"^\s*```(?:python)?\s*\n?", "", response, count=1)
  # Remove trailing '```'
  response = re.sub(r"\n?\s*```\s*$", "", response, count=1)

  steps = ast.literal_eval(response)
  return steps

In [19]:
def research_agent(task, context=""):
  prompt = f"""
  You are a research agent. Your job is to research the following task using
  your knowledge and provide detailed, factual information.

  Task: {task}

  Today's data: {datetime.now().strftime('%Y-%m-%d')}

  Provide a detailed response with relevant facts, findings, and information
  related to this task. Be specific and informative.
  """

  response = get_completion(prompt, temperature=0.7)
  return response

In [20]:
def writer_agent(task, context=""):
  prompt = f"""
  You are a writer agent specialized in generating well-structured, professional
  research content.

  Here is the context from previous research:
  {context}

  Your task:
  {task}

  Write clear, well-organized content in Markdown format.
  """
  response = get_completion(prompt, temperature=1.0)
  return response


In [21]:
def editor_agent(task, context=""):
  prompt = f"""
  You are an editor agent specialized in reflecting on, critiquing,
  and improving research drafts.

  Here is the draft content to review and improve:
  ---
  {context}
  ---

  Your task: {task}

  IMPORTANT: Do NOT explain your editing process or give instructions.
  Directly return the COMPLETE, IMPROVED version of the content above,
  in Markdown format. Output ONLY the revised content.
  """
  response = get_completion(prompt, temperature=0.7)
  return response

In [22]:
def research_pipeline(topic):
    print(f"📋 Creating plan for: {topic}\n")

    # Step 1: Get the plan
    plan = planner_agent(topic)

    for i, step in enumerate(plan):
        print(f"Step {i+1}: {step}")
    print("\n" + "="*50 + "\n")

    # Step 2: Set up agent registry
    agent_registry = {
        "research_agent": research_agent,
        "writer_agent": writer_agent,
        "editor_agent": editor_agent,
    }

    context = ""
    history = []

    # Step 3: Loop through each step
    for i, step in enumerate(plan):

        # Smart routing — check step text first!
        step_lower = step.lower()
        if "research agent" in step_lower:
            agent_name = "research_agent"
        elif "writer agent" in step_lower:
            agent_name = "writer_agent"
        elif "editor agent" in step_lower:
            agent_name = "editor_agent"
        else:
            # Fall back to AI router
            decision_prompt = f"""
            You are a workflow manager. Given this instruction, decide which agent should handle it.

            Agents available:
            - "research_agent": for searching/gathering information
            - "writer_agent": for drafting, summarizing, or writing content
            - "editor_agent": for reviewing, revising, or finalizing content

            Instruction: "{step}"

            Return ONLY the agent name, nothing else.
            """
            agent_name = get_completion(decision_prompt, temperature=0).strip()
            agent_name = agent_name.replace('"', '').replace("'", "").strip()

        print(f"🛠️ Step {i+1}: Using {agent_name}")
        print(f"   Task: {step}\n")

        # Call the chosen agent
        if agent_name in agent_registry:
            output = agent_registry[agent_name](step, context=context)
        else:
            output = research_agent(step, context=context)

        history.append((step, agent_name, output))
        context = output

        print(f"✅ Output preview: {output[:200]}...\n")
        print("-"*50 + "\n")

    return history

In [ ]:
result = research_pipeline("New Advancements about Black holes")

print("\n" + "="*50)
print("📄 FINAL REPORT")
print("="*50 + "\n")

final_report = result[-1][-1]
print(final_report)